# Fase 2 y Fase 6: Hipotesis, pruebas estadisticas y validacion

Este notebook formula las hipotesis del proyecto con `H0` y `H1`, justifica su relevancia de negocio, ejecuta las pruebas estadisticas y guarda una matriz final en `reports/resultados_hipotesis.csv`.


In [ ]:
# Importamos librerias para analisis, pruebas estadisticas y graficos.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Definimos rutas y cargamos la base limpia preparada en el notebook 01.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
VIS_DIR = PROJECT_ROOT / "dashboard" / "visualizaciones"
df = pd.read_csv(DATA_DIR / "clean_hr_analytics.csv")

# Dejamos un estilo simple para las figuras.
sns.set_theme(style="whitegrid")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset limpio: {df.shape[0]:,} registros, {df.shape[1]} variables")


## Distribucion general de rotacion

In [ ]:
# Resumimos cuantos empleados se quedan y cuantos se van.
rotation_counts = df["left_label"].value_counts().rename_axis("estado").reset_index(name="empleados")
rotation_pct = df["left_label"].value_counts(normalize=True).mul(100).round(2).rename_axis("estado").reset_index(name="porcentaje")

# Unimos conteos y porcentajes para leer el panorama general.
rotation_summary = rotation_counts.merge(rotation_pct, on="estado")
display(rotation_summary)

In [ ]:
# Dibujamos una grafica sencilla para ver la distribucion de rotacion.
plt.figure(figsize=(6, 4))
sns.barplot(data=rotation_summary, x="estado", y="empleados", palette=["#4c8c6a", "#c44e52"])
plt.title("Distribucion general de rotacion")
plt.xlabel("Estado")
plt.ylabel("Empleados")
plt.show()

## Hipotesis 1: horas mensuales vs satisfaccion

- Variables: `average_monthly_hours` y `satisfaction_level`.
- H0: No existe relacion lineal estadisticamente significativa entre las horas mensuales promedio y el nivel de satisfaccion laboral.
- H1: Existe una relacion lineal negativa estadisticamente significativa entre las horas mensuales promedio y el nivel de satisfaccion laboral.
- Justificacion de negocio: si una mayor carga horaria se asocia con menor satisfaccion, Recursos Humanos puede intervenir la distribucion de carga, prevenir desgaste y reducir riesgo de salida.


In [ ]:
# H1 revisa si existe relacion lineal entre horas mensuales y satisfaccion.
corr, p_value = stats.pearsonr(df["average_monthly_hours"], df["satisfaction_level"])
h1_decision = "Rechazar H0" if p_value < 0.05 else "No rechazar H0"

print(f"Coeficiente r: {corr:.4f}")
print(f"Valor p: {p_value:.6f}")
print("Decision:", h1_decision)

In [ ]:
# Graficamos una muestra para ver la tendencia sin saturar la figura.
sample = df.sample(min(len(df), 3000), random_state=42)
plt.figure(figsize=(8, 5))
sns.regplot(
    data=sample,
    x="average_monthly_hours",
    y="satisfaction_level",
    scatter_kws={"alpha": 0.25, "s": 14, "color": "steelblue"},
    line_kws={"color": "red"},
)
plt.title("H1: horas mensuales vs satisfaccion")
plt.xlabel("Horas mensuales promedio")
plt.ylabel("Satisfaccion")
plt.annotate(
    f"r = {corr:.2f} | p = {p_value:.3f}",
    xy=(0.05, 0.95),
    xycoords="axes fraction",
    fontsize=10,
    va="top",
)
plt.tight_layout()
plt.savefig(VIS_DIR / "h1_horas_vs_satisfaccion.png", dpi=150, bbox_inches="tight")
plt.show()


## Hipotesis 2: satisfaccion baja vs rotacion

- Variables: `satisfaction_category` y `left`.
- H0: La rotacion es independiente de la categoria de satisfaccion del empleado.
- H1: La rotacion depende de la categoria de satisfaccion y la salida es mayor en empleados con satisfaccion baja.
- Justificacion de negocio: permite priorizar acciones de clima laboral y retencion en el grupo con mayor riesgo de abandono.


In [ ]:
# H2 revisa si la satisfaccion baja esta asociada con la salida del empleado.
tabla_sat = pd.crosstab(df["satisfaction_category"], df["left"])
chi2_sat, p_sat, dof_sat, _ = stats.chi2_contingency(tabla_sat)
h2_decision = "Rechazar H0" if p_sat < 0.05 else "No rechazar H0"

display(tabla_sat)
print(f"Chi-cuadrado: {chi2_sat:.4f}")
print(f"Grados de libertad: {dof_sat}")
print(f"Valor p: {p_sat:.6f}")
print("Decision:", h2_decision)

In [ ]:
# Convertimos a porcentaje para comparar permanencia y salida dentro de cada grupo.
sat_pct = pd.crosstab(df["satisfaction_category"], df["left_label"], normalize="index").mul(100)
sat_pct[["Permanece", "Se va"]].plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=["steelblue", "tomato"],
)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", label_type="center", fontsize=8)
plt.title("H2: rotacion por nivel de satisfaccion")
plt.xlabel("Categoria de satisfaccion")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=0)
plt.legend(["Permanece (0)", "Se va (1)"])
plt.tight_layout()
plt.savefig(VIS_DIR / "h2_satisfaccion_vs_rotacion.png", dpi=150, bbox_inches="tight")
plt.show()


## Hipotesis 3: salario vs rotacion

- Variables: `salary_label` y `left`.
- H0: La rotacion es independiente del nivel salarial.
- H1: La rotacion depende del nivel salarial y es mayor en empleados con salario bajo.
- Justificacion de negocio: orienta decisiones de compensacion y revisiones salariales para disminuir salida de talento.


In [ ]:
# H3 revisa si el nivel salarial esta asociado con la rotacion.
tabla_salary = pd.crosstab(df["salary_label"], df["left"])
chi2_salary, p_salary, dof_salary, _ = stats.chi2_contingency(tabla_salary)
h3_decision = "Rechazar H0" if p_salary < 0.05 else "No rechazar H0"

display(tabla_salary)
print(f"Chi-cuadrado: {chi2_salary:.4f}")
print(f"Grados de libertad: {dof_salary}")
print(f"Valor p: {p_salary:.6f}")
print("Decision:", h3_decision)

In [ ]:
# Pasamos a porcentaje para comparar cada nivel salarial.
salary_pct = pd.crosstab(df["salary_label"], df["left_label"], normalize="index").mul(100)
salary_pct.reindex(["Bajo", "Medio", "Alto"])[["Permanece", "Se va"]].plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=["steelblue", "tomato"],
)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", label_type="center", fontsize=8)
plt.title("H3: rotacion por nivel salarial")
plt.xlabel("Nivel salarial")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=0)
plt.legend(["Permanece (0)", "Se va (1)"])
plt.tight_layout()
plt.savefig(VIS_DIR / "h3_salario_vs_rotacion.png", dpi=150, bbox_inches="tight")
plt.show()


## Hipotesis 4: departamentos con mas horas vs mayor rotacion

- Variables: `department_label`, `average_monthly_hours` y `left` agregadas por departamento.
- H0: No existe una relacion monotona estadisticamente significativa entre las horas promedio departamentales y la rotacion departamental.
- H1: Existe una relacion monotona positiva estadisticamente significativa entre las horas promedio departamentales y la rotacion departamental.
- Justificacion de negocio: valida si la sobrecarga operativa por area explica la salida y si conviene intervenir departamentos especificos.


In [ ]:
# H4 usa promedios exactos por departamento antes de redondear.
department = (
    df.groupby(["department", "department_label"], as_index=False)
    .agg(
        employees=("employee_count", "sum"),
        avg_monthly_hours=("average_monthly_hours", "mean"),
        turnover_rate=("left", "mean"),
        avg_satisfaction=("satisfaction_level", "mean"),
    )
    .sort_values("turnover_rate", ascending=False)
)
department["turnover_rate_pct"] = department["turnover_rate"].mul(100)
spearman, p_dept = stats.spearmanr(department["avg_monthly_hours"], department["turnover_rate"])
h4_decision = "Rechazar H0" if p_dept < 0.05 else "No rechazar H0"

display(
    department[["department_label", "employees", "avg_monthly_hours", "turnover_rate_pct", "avg_satisfaction"]]
    .round({"avg_monthly_hours": 1, "turnover_rate_pct": 2, "avg_satisfaction": 3})
)
print(f"Spearman rho: {spearman:.4f}")
print(f"Valor p: {p_dept:.6f}")
print("Decision:", h4_decision)

In [ ]:
# Graficamos cada departamento para comparar horas promedio y rotacion.
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=department,
    x="avg_monthly_hours",
    y="turnover_rate_pct",
    size="employees",
    hue="avg_satisfaction",
    sizes=(80, 350),
)
for _, row in department.iterrows():
    plt.text(row["avg_monthly_hours"] + 0.2, row["turnover_rate_pct"], row["department_label"], fontsize=8)
plt.title("H4: horas promedio y rotacion por departamento")
plt.xlabel("Horas mensuales promedio")
plt.ylabel("Rotacion (%)")
plt.show()

## Exportacion de resultados y matriz de hipotesis


In [ ]:
# Guardamos una matriz final con las hipotesis, su justificacion y sus resultados.
resultados = pd.DataFrame(
    [
        [
            "H1",
            "Horas mensuales vs satisfaccion",
            "average_monthly_hours, satisfaction_level",
            "No existe relacion lineal estadisticamente significativa entre las horas mensuales promedio y el nivel de satisfaccion laboral.",
            "Existe una relacion lineal negativa estadisticamente significativa entre las horas mensuales promedio y el nivel de satisfaccion laboral.",
            "Permite evaluar si la carga de trabajo debe monitorearse como factor de desgaste y retencion.",
            "Correlacion de Pearson",
            round(float(corr), 4),
            round(float(p_value), 6),
            None,
            h1_decision,
            "La relacion lineal es negativa pero muy debil; las horas por si solas no explican la satisfaccion.",
        ],
        [
            "H2",
            "Satisfaccion baja vs rotacion",
            "satisfaction_category, left",
            "La rotacion es independiente de la categoria de satisfaccion del empleado.",
            "La rotacion depende de la categoria de satisfaccion y la salida es mayor en empleados con satisfaccion baja.",
            "Ayuda a priorizar programas de clima laboral para grupos con mayor riesgo de salida.",
            "Chi-cuadrado de independencia",
            round(float(chi2_sat), 4),
            round(float(p_sat), 6),
            int(dof_sat),
            h2_decision,
            "Existe asociacion entre satisfaccion baja y rotacion.",
        ],
        [
            "H3",
            "Salario vs rotacion",
            "salary_label, left",
            "La rotacion es independiente del nivel salarial.",
            "La rotacion depende del nivel salarial y es mayor en empleados con salario bajo.",
            "Orienta decisiones de compensacion y focalizacion salarial.",
            "Chi-cuadrado de independencia",
            round(float(chi2_salary), 4),
            round(float(p_salary), 6),
            int(dof_salary),
            h3_decision,
            "Existe asociacion entre nivel salarial y rotacion.",
        ],
        [
            "H4",
            "Horas promedio departamentales vs rotacion departamental",
            "department_label, average_monthly_hours, left",
            "No existe una relacion monotona estadisticamente significativa entre las horas promedio departamentales y la rotacion departamental.",
            "Existe una relacion monotona positiva estadisticamente significativa entre las horas promedio departamentales y la rotacion departamental.",
            "Permite validar si la sobrecarga por departamento explica la salida y si conviene intervenir equipos especificos.",
            "Correlacion de Spearman",
            round(float(spearman), 4),
            round(float(p_dept), 6),
            None,
            h4_decision,
            "No se observa evidencia fuerte de que los departamentos con mas horas promedio sean necesariamente los de mayor rotacion.",
        ],
    ],
    columns=[
        "hipotesis",
        "tema",
        "variables",
        "h0",
        "h1",
        "justificacion_negocio",
        "prueba",
        "estadistico",
        "valor_p",
        "grados_libertad",
        "decision",
        "interpretacion",
    ],
)

resultados.to_csv(REPORTS_DIR / "resultados_hipotesis.csv", index=False, encoding="utf-8")
display(resultados)
